In [11]:
# Build main diff-in-diff analysis data

import os
import pandas as pd
import numpy as np
import dotenv

import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

dotenv.load_dotenv(dotenv.find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")

OUTPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data/tax_analysis_panel.parquet")


In [12]:
# Load data

regs_df = pd.read_csv(os.path.join(RAW_DATA_PATH, "sales-analysis-redfin/data/best_treatment_dates_2026-07.csv"))
fisc_df = pd.read_excel(os.path.join(RAW_DATA_PATH, "lincoln-institute/FiSC-Full-Dataset-2023-Update.xlsx"), sheet_name="Data")
zhvi_df = pd.read_csv(os.path.join(RAW_DATA_PATH, "zhvi/City_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"))
alfin_df = pd.read_parquet(os.path.join(MY_DATA_PATH, "processed_data/alfin_panel.parquet"))

zhvi_xwalk = pd.read_csv(os.path.join(MY_DATA_PATH, "raw_data/zhvi_xwalk.csv"))
fisc_xwalk = pd.read_csv(os.path.join(MY_DATA_PATH, "raw_data/fisc_xwalk.csv"))


In [13]:
# Reshape zhvi data long by year

id_cols = ['RegionID', 'SizeRank', 'RegionName', 'RegionType',
           'StateName', 'State', 'Metro', 'CountyName']

zhvi_long = zhvi_df.melt(
    id_vars=id_cols,
    var_name = 'date',
    value_name = 'ZHVI'
)

zhvi_long['date'] = pd.to_datetime(zhvi_long['date'])
zhvi_long['year'] = zhvi_long['date'].dt.year

zhvi_long = zhvi_long.groupby(id_cols + ['year']).agg({'ZHVI': 'mean'}).reset_index()


In [14]:
# Merging the data

df = regs_df.merge(fisc_xwalk, on=['city', 'state'], how='left')
df = df.merge(zhvi_xwalk, on=['city', 'state'], how='left')

df = df.merge(alfin_df, on=['city', 'state'], how='left')
df = df.merge(fisc_df.rename(columns={'city_name': 'fisc_city'}), on=['fisc_city', 'year'], how='left')
df = df.merge(zhvi_long[['RegionID', 'year', 'ZHVI']], on=['RegionID', 'year'], how='left')


In [15]:
assert df[['city', 'state', 'year']].duplicated().sum() == 0
n_city = len(df.groupby(['city', 'state']))
n_year = len(df['year'].unique())
assert len(df) == n_city * n_year

In [16]:
# clean dates

df['best_enforcement'] = pd.to_datetime(df['best_enforcement'], errors='coerce')
df['best_passage'] = pd.to_datetime(df['best_passage'], errors='coerce')

df['enforcement_year'] = df['best_enforcement'].dt.year
df['passage_year'] = df['best_passage'].dt.year

df['years_from_enforcement'] = (df['year'] - df['enforcement_year'])
df['years_from_passage'] = (df['year'] - df['passage_year'])


In [17]:
# change enforcement and passage year to 0 for cities without enforcement/passage dates
# (standard convention for CSDID package in R)

df.loc[df['best_enforcement'].isna(), 'enforcement_year'] = 0
df.loc[df['best_passage'].isna(), 'passage_year'] = 0

df['enforcement_year'] = df['enforcement_year'].astype(int)
df['passage_year'] = df['passage_year'].astype(int)


In [18]:
# make a city_id integer (also required for CSDID package)

df['city_id'] = df['city'].astype('category').cat.codes

In [19]:
# output dataframe for analysis

df.to_parquet(OUTPUT_FILEPATH)

df['year'].value_counts()

year
2007    50
2008    50
2009    50
2010    50
2011    50
2012    50
2013    50
2014    50
2015    50
2016    50
2017    50
2018    50
2019    50
2020    50
2021    50
2022    50
2023    50
Name: count, dtype: int64